# 01 - SQL Traffic Analysis with SQLite

This learning notebook covers **Part 1, Task 1** of the capstone. It loads the supplied traffic CSV into an in-memory SQLite database, verifies the import, analyses annual traffic, and compares temperatures around New Year's Day and Labor Day.

## Learning objectives

1. Create a typed SQLite table.
2. Import CSV records safely.
3. Verify row count, schema, nulls and duplicates.
4. Use `GROUP BY`, date functions, common table expressions and `LAG`.
5. Compare raw-record totals with one-record-per-hour totals.
6. Identify limitations in annual and holiday comparisons.

> This notebook uses `:memory:` SQLite. Closing the notebook removes the learning database. No final `.db` or `.sql` deliverable is created yet.

## 1. Import standard-library tools and locate the CSV

Python includes SQLite through the `sqlite3` module, so this notebook does not need an additional database package.

In [1]:
import csv
import sqlite3
from pathlib import Path

CSV_FILENAME = "Metro_Interstate_Traffic_Volume.csv"

def find_project_root(start: Path, filename: str) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / filename).is_file():
            return candidate
    raise FileNotFoundError(f"Could not find {filename!r}.")

PROJECT_ROOT = find_project_root(Path.cwd(), CSV_FILENAME)
CSV_PATH = PROJECT_ROOT / CSV_FILENAME

print(f"SQLite version: {sqlite3.sqlite_version}")
print(f"CSV file: {CSV_PATH.name}")
print(f"CSV size: {CSV_PATH.stat().st_size:,} bytes")

SQLite version: 3.45.1
CSV file: Metro_Interstate_Traffic_Volume.csv
CSV size: 3,237,208 bytes


## 2. Create a query-display helper

The helper executes a read-only query and prints a small table. The final submission will store the SQL statements in `.sql` files rather than hiding them inside Python.

In [2]:
def run_query(connection, sql, parameters=()):
    cursor = connection.execute(sql, parameters)
    headers = [item[0] for item in cursor.description]
    records = cursor.fetchall()
    text_rows = [["NULL" if value is None else str(value) for value in row] for row in records]
    widths = [len(header) for header in headers]
    for row in text_rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]
    print(" | ".join(header.ljust(width) for header, width in zip(headers, widths)))
    print("-+-".join("-" * width for width in widths))
    for row in text_rows:
        print(" | ".join(value.ljust(width) for value, width in zip(row, widths)))
    print(f"\n{len(records)} row(s) returned.")
    return records

## 3. Create a typed traffic table

The table uses text for categorical and ISO-formatted date/time values, `REAL` for measurements, and `INTEGER` for whole-number values. `row_id` preserves the source-row identity even when two records otherwise match.

In [3]:
connection = sqlite3.connect(":memory:")
connection.execute("PRAGMA foreign_keys = ON")

connection.execute(
    """
    CREATE TABLE traffic (
        row_id INTEGER PRIMARY KEY,
        holiday TEXT NOT NULL,
        temp REAL NOT NULL,
        rain_1h REAL NOT NULL,
        snow_1h REAL NOT NULL,
        clouds_all INTEGER NOT NULL,
        weather_main TEXT NOT NULL,
        weather_description TEXT NOT NULL,
        date_time TEXT NOT NULL,
        traffic_volume INTEGER NOT NULL
    )
    """
)

print("Created in-memory table: traffic")

Created in-memory table: traffic


## 4. Convert and import the CSV records

Values are explicitly converted before insertion. A malformed number will therefore produce a useful line-specific error instead of silently entering the database as the wrong type.

In [4]:
records_to_insert = []

with CSV_PATH.open(mode="r", encoding="utf-8-sig", newline="") as csv_file:
    reader = csv.DictReader(csv_file)
    for csv_line_number, row in enumerate(reader, start=2):
        try:
            records_to_insert.append(
                (
                    row["holiday"],
                    float(row["temp"]),
                    float(row["rain_1h"]),
                    float(row["snow_1h"]),
                    int(row["clouds_all"]),
                    row["weather_main"],
                    row["weather_description"],
                    row["date_time"],
                    int(row["traffic_volume"]),
                )
            )
        except (KeyError, TypeError, ValueError) as exc:
            raise ValueError(f"Invalid data on CSV line {csv_line_number}: {exc}") from exc

with connection:
    connection.executemany(
        """
        INSERT INTO traffic (
            holiday, temp, rain_1h, snow_1h, clouds_all,
            weather_main, weather_description, date_time, traffic_volume
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        records_to_insert,
    )
    connection.execute("CREATE INDEX idx_traffic_date_time ON traffic(date_time)")
    connection.execute("CREATE INDEX idx_traffic_holiday ON traffic(holiday)")

print(f"Inserted {len(records_to_insert):,} records into SQLite.")

Inserted 48,204 records into SQLite.


## 5. Verify the imported schema and data

Import verification should check more than whether the command completed. We inspect the table definition, row count, date range, unique timestamps and nulls.

In [5]:
schema_rows = run_query(connection, "PRAGMA table_info(traffic)")

cid | name                | type    | notnull | dflt_value | pk
----+---------------------+---------+---------+------------+---
0   | row_id              | INTEGER | 0       | NULL       | 1 
1   | holiday             | TEXT    | 1       | NULL       | 0 
2   | temp                | REAL    | 1       | NULL       | 0 
3   | rain_1h             | REAL    | 1       | NULL       | 0 
4   | snow_1h             | REAL    | 1       | NULL       | 0 
5   | clouds_all          | INTEGER | 1       | NULL       | 0 
6   | weather_main        | TEXT    | 1       | NULL       | 0 
7   | weather_description | TEXT    | 1       | NULL       | 0 
8   | date_time           | TEXT    | 1       | NULL       | 0 
9   | traffic_volume      | INTEGER | 1       | NULL       | 0 

10 row(s) returned.


In [6]:
verification_rows = run_query(
    connection,
    """
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT date_time) AS unique_timestamps,
        MIN(date_time) AS earliest_timestamp,
        MAX(date_time) AS latest_timestamp,
        SUM(
            CASE WHEN holiday IS NULL OR temp IS NULL OR rain_1h IS NULL
                OR snow_1h IS NULL OR clouds_all IS NULL
                OR weather_main IS NULL OR weather_description IS NULL
                OR date_time IS NULL OR traffic_volume IS NULL
            THEN 1 ELSE 0 END
        ) AS rows_with_nulls
    FROM traffic
    """,
)

row_count | unique_timestamps | earliest_timestamp  | latest_timestamp    | rows_with_nulls
----------+-------------------+---------------------+---------------------+----------------
48204     | 40575             | 2012-10-02 09:00:00 | 2018-09-30 23:00:00 | 0              

1 row(s) returned.


## 6. Distinguish exact duplicates from repeated hours

The source has 17 exact duplicate rows, but many more repeated timestamps represent multiple weather descriptions for the same hour. The second query checks whether repeated timestamps disagree about traffic volume.

In [7]:
duplicate_rows = run_query(
    connection,
    """
    SELECT COALESCE(SUM(group_size - 1), 0) AS exact_duplicate_rows
    FROM (
        SELECT COUNT(*) AS group_size
        FROM traffic
        GROUP BY
            holiday, temp, rain_1h, snow_1h, clouds_all,
            weather_main, weather_description, date_time, traffic_volume
        HAVING COUNT(*) > 1
    )
    """,
)

timestamp_consistency_rows = run_query(
    connection,
    """
    SELECT COUNT(*) AS timestamps_with_conflicting_traffic
    FROM (
        SELECT date_time
        FROM traffic
        GROUP BY date_time
        HAVING MIN(traffic_volume) <> MAX(traffic_volume)
    )
    """,
)

exact_duplicate_rows
--------------------
17                  

1 row(s) returned.
timestamps_with_conflicting_traffic
-----------------------------------
0                                  

1 row(s) returned.


## 7. Inspect annual data coverage

A total is affected by the amount of data available. This query must be read before interpreting year-to-year totals.

In [8]:
coverage_rows = run_query(
    connection,
    """
    SELECT
        CAST(strftime('%Y', date_time) AS INTEGER) AS year,
        COUNT(*) AS source_records,
        COUNT(DISTINCT date_time) AS unique_hours,
        MIN(date_time) AS first_record,
        MAX(date_time) AS last_record
    FROM traffic
    WHERE CAST(strftime('%Y', date_time) AS INTEGER) BETWEEN 2012 AND 2017
    GROUP BY year
    ORDER BY year
    """,
)

year | source_records | unique_hours | first_record        | last_record        
-----+----------------+--------------+---------------------+--------------------
2012 | 2559           | 2103         | 2012-10-02 09:00:00 | 2012-12-31 23:00:00
2013 | 8573           | 7294         | 2013-01-01 00:00:00 | 2013-12-31 23:00:00
2014 | 4839           | 4501         | 2014-01-01 00:00:00 | 2014-08-08 01:00:00
2015 | 4373           | 3593         | 2015-06-11 20:00:00 | 2015-12-31 23:00:00
2016 | 9306           | 7838         | 2016-01-01 00:00:00 | 2016-12-31 23:00:00
2017 | 10605          | 8713         | 2017-01-01 00:00:00 | 2017-12-31 23:00:00

6 row(s) returned.


## 8. Calculate requested annual totals and changes

`LAG` retrieves the preceding year's total. The raw-record result follows a direct interpretation of the assignment, but hours with multiple weather descriptions contribute their traffic volume more than once.

In [9]:
raw_annual_rows = run_query(
    connection,
    """
    WITH annual AS (
        SELECT
            CAST(strftime('%Y', date_time) AS INTEGER) AS year,
            COUNT(*) AS source_records,
            COUNT(DISTINCT date_time) AS unique_hours,
            SUM(traffic_volume) AS total_traffic_volume,
            ROUND(AVG(traffic_volume), 2) AS average_traffic_volume
        FROM traffic
        WHERE CAST(strftime('%Y', date_time) AS INTEGER) BETWEEN 2012 AND 2017
        GROUP BY year
    ),
    annual_with_previous AS (
        SELECT
            *,
            LAG(total_traffic_volume) OVER (ORDER BY year) AS previous_total
        FROM annual
    )
    SELECT
        year, source_records, unique_hours, total_traffic_volume,
        average_traffic_volume,
        total_traffic_volume - previous_total AS year_on_year_change,
        CASE
            WHEN previous_total IS NULL THEN NULL
            ELSE ROUND(
                100.0 * (total_traffic_volume - previous_total) / previous_total,
                2
            )
        END AS year_on_year_percent
    FROM annual_with_previous
    ORDER BY year
    """,
)

year | source_records | unique_hours | total_traffic_volume | average_traffic_volume | year_on_year_change | year_on_year_percent
-----+----------------+--------------+----------------------+------------------------+---------------------+---------------------
2012 | 2559           | 2103         | 8208767              | 3207.8                 | NULL                | NULL                
2013 | 8573           | 7294         | 28177412             | 3286.76                | 19968645            | 243.26              
2014 | 4839           | 4501         | 15731289             | 3250.94                | -12446123           | -44.17              
2015 | 4373           | 3593         | 14181206             | 3242.9                 | -1550083            | -9.85               
2016 | 9306           | 7838         | 29494821             | 3169.44                | 15313615            | 107.99              
2017 | 10605          | 8713         | 35428156             | 3340.7                 | 593

## 9. Compare with one-record-per-hour totals

Because all repeated timestamps agree on traffic volume, `MAX(traffic_volume)` safely selects the single hourly value. This avoids counting one hour several times merely because several weather descriptions were recorded. Both methods should be documented rather than silently choosing one.

In [10]:
connection.execute(
    """
    CREATE TEMP VIEW hourly_traffic AS
    SELECT
        date_time,
        MAX(traffic_volume) AS traffic_volume
    FROM traffic
    GROUP BY date_time
    """
)

hourly_annual_rows = run_query(
    connection,
    """
    WITH annual AS (
        SELECT
            CAST(strftime('%Y', date_time) AS INTEGER) AS year,
            COUNT(*) AS unique_hours,
            SUM(traffic_volume) AS hourly_total_traffic_volume,
            ROUND(AVG(traffic_volume), 2) AS average_hourly_traffic
        FROM hourly_traffic
        WHERE CAST(strftime('%Y', date_time) AS INTEGER) BETWEEN 2012 AND 2017
        GROUP BY year
    ),
    annual_with_previous AS (
        SELECT
            *,
            LAG(hourly_total_traffic_volume) OVER (ORDER BY year) AS previous_total
        FROM annual
    )
    SELECT
        year, unique_hours, hourly_total_traffic_volume, average_hourly_traffic,
        hourly_total_traffic_volume - previous_total AS year_on_year_change,
        CASE
            WHEN previous_total IS NULL THEN NULL
            ELSE ROUND(
                100.0 * (hourly_total_traffic_volume - previous_total) / previous_total,
                2
            )
        END AS year_on_year_percent
    FROM annual_with_previous
    ORDER BY year
    """,
)

year | unique_hours | hourly_total_traffic_volume | average_hourly_traffic | year_on_year_change | year_on_year_percent
-----+--------------+-----------------------------+------------------------+---------------------+---------------------
2012 | 2103         | 6785754                     | 3226.7                 | NULL                | NULL                
2013 | 7294         | 24139878                    | 3309.55                | 17354124            | 255.74              
2014 | 4501         | 14718915                    | 3270.14                | -9420963            | -39.03              
2015 | 3593         | 11706145                    | 3258.04                | -3012770            | -20.47              
2016 | 7838         | 25032183                    | 3193.7                 | 13326038            | 113.84              
2017 | 8713         | 29420221                    | 3376.59                | 4388038             | 17.53               

6 row(s) returned.


## 10. Inspect the holiday-tagged records

The holiday field is usually populated only at midnight. Some holiday timestamps appear more than once because of multiple weather descriptions. The left-joined grid makes missing year/holiday combinations visible instead of dropping them from the result.

In [11]:
holiday_tag_rows = run_query(
    connection,
    """
    WITH years(year) AS (VALUES (2015), (2016), (2017)),
    holidays(holiday) AS (VALUES ('New Years Day'), ('Labor Day')),
    expected AS (
        SELECT year, holiday FROM years CROSS JOIN holidays
    )
    SELECT
        expected.year,
        expected.holiday,
        COUNT(traffic.row_id) AS source_records,
        COUNT(DISTINCT traffic.date_time) AS unique_timestamps,
        ROUND(AVG(traffic.temp), 2) AS average_temp_kelvin,
        ROUND(AVG(traffic.traffic_volume), 2) AS average_traffic_volume
    FROM expected
    LEFT JOIN traffic
        ON CAST(strftime('%Y', traffic.date_time) AS INTEGER) = expected.year
        AND traffic.holiday = expected.holiday
    GROUP BY expected.year, expected.holiday
    ORDER BY expected.year, expected.holiday DESC
    """,
)

year | holiday       | source_records | unique_timestamps | average_temp_kelvin | average_traffic_volume
-----+---------------+----------------+-------------------+---------------------+-----------------------
2015 | New Years Day | 0              | 0                 | NULL                | NULL                  
2015 | Labor Day     | 1              | 1                 | 295.02              | 973.0                 
2016 | New Years Day | 2              | 1                 | 265.94              | 1513.0                
2016 | Labor Day     | 2              | 1                 | 293.17              | 1064.0                
2017 | New Years Day | 1              | 1                 | 270.62              | 798.0                 
2017 | Labor Day     | 1              | 1                 | 295.54              | 1026.0                

6 row(s) returned.


## 11. Analyse the full observed holiday dates

To discuss a temperature *pattern* rather than one midnight measurement, this query first discovers each observed holiday date and then summarises all available hours on that date. It consolidates repeated weather observations to one row per timestamp before calculating daily values.

Remember that the dataset labels the observed federal holiday. For example, the 2017 New Year's holiday appears on 2 January.

In [12]:
holiday_day_rows = run_query(
    connection,
    """
    WITH holiday_dates AS (
        SELECT DISTINCT
            CAST(strftime('%Y', date_time) AS INTEGER) AS year,
            holiday,
            date(date_time) AS holiday_date
        FROM traffic
        WHERE holiday IN ('New Years Day', 'Labor Day')
          AND CAST(strftime('%Y', date_time) AS INTEGER) BETWEEN 2015 AND 2017
    ),
    hourly_holiday_data AS (
        SELECT
            holiday_dates.year,
            holiday_dates.holiday,
            holiday_dates.holiday_date,
            traffic.date_time,
            AVG(traffic.temp) AS hourly_temp_kelvin,
            MAX(traffic.traffic_volume) AS hourly_traffic_volume
        FROM holiday_dates
        JOIN traffic ON date(traffic.date_time) = holiday_dates.holiday_date
        GROUP BY
            holiday_dates.year, holiday_dates.holiday,
            holiday_dates.holiday_date, traffic.date_time
    )
    SELECT
        year, holiday, holiday_date,
        COUNT(*) AS observed_hours,
        ROUND(MIN(hourly_temp_kelvin), 2) AS minimum_temp_kelvin,
        ROUND(AVG(hourly_temp_kelvin), 2) AS average_temp_kelvin,
        ROUND(MAX(hourly_temp_kelvin), 2) AS maximum_temp_kelvin,
        ROUND(AVG(hourly_traffic_volume), 2) AS average_traffic_volume
    FROM hourly_holiday_data
    GROUP BY year, holiday, holiday_date
    ORDER BY holiday, year
    """,
)

year | holiday       | holiday_date | observed_hours | minimum_temp_kelvin | average_temp_kelvin | maximum_temp_kelvin | average_traffic_volume
-----+---------------+--------------+----------------+---------------------+---------------------+---------------------+-----------------------
2015 | Labor Day     | 2015-09-07   | 24             | 291.85              | 295.32              | 299.6               | 2430.29               
2016 | Labor Day     | 2016-09-05   | 24             | 291.88              | 295.01              | 297.99              | 2151.54               
2017 | Labor Day     | 2017-09-04   | 24             | 284.93              | 291.29              | 296.22              | 2395.17               
2016 | New Years Day | 2016-01-01   | 18             | 263.99              | 267.09              | 271.98              | 1832.0                
2017 | New Years Day | 2017-01-02   | 24             | 270.42              | 271.85              | 273.02              | 2091.08        

## 12. Run automated SQL checks

These assertions verify the imported source and important query results. They do not assert that incomplete years are comparable; that remains an analytical limitation to explain in the report.

In [13]:
row_count, unique_timestamps, earliest, latest, rows_with_nulls = verification_rows[0]

assert row_count == 48_204
assert unique_timestamps == 40_575
assert earliest == "2012-10-02 09:00:00"
assert latest == "2018-09-30 23:00:00"
assert rows_with_nulls == 0
assert duplicate_rows[0][0] == 17
assert timestamp_consistency_rows[0][0] == 0
assert [row[0] for row in raw_annual_rows] == [2012, 2013, 2014, 2015, 2016, 2017]
assert [row[3] for row in raw_annual_rows] == [
    8_208_767, 28_177_412, 15_731_289,
    14_181_206, 29_494_821, 35_428_156,
]

holiday_lookup = {(row[0], row[1]): row for row in holiday_tag_rows}
assert holiday_lookup[(2015, "New Years Day")][2] == 0
assert holiday_lookup[(2015, "Labor Day")][4] == 295.02
assert holiday_lookup[(2016, "New Years Day")][4] == 265.94
assert holiday_lookup[(2017, "New Years Day")][4] == 270.62

print("PASS - SQLite import and Part 1 Task 1 queries matched expected results.")

PASS - SQLite import and Part 1 Task 1 queries matched expected results.


## 13. Interpretation checkpoint

Before creating final SQL files, be able to explain:

1. Why 2012, 2014 and 2015 totals are especially affected by incomplete coverage.
2. Why raw-record totals exceed one-record-per-hour totals.
3. Why the 2015 New Year's comparison must be reported as unavailable.
4. Why weather and temperature may be associated with traffic but are not necessarily causal.
5. Which annual-total definition you will present and how you will disclose its limitation.

### Stop point

Run all cells and confirm the final code cell reports `PASS`. Save the notebook, then we will review the SQL results before producing the graded `.sql` file and SQLite database.